# Choose a loss function and regularization for your data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-hahn/bayesian-models-of-perception/blob/main/code/Demo/Fit_Your_Own_Data.ipynb)

This is a run-from-top-to-bottom tutorial. You will fit one baseline model, compare several candidate decision-loss functions, repeat the key comparison across five sensory-noise levels, tune smoothness regularization, confirm a shortlist across held-out folds, and inspect the inferred prior and encoding allocation. Run each code cell when the text asks you to; the notebook makes the comparisons for you.

The central lesson is that the assumed loss function affects the Bayes action. A wrong loss can therefore be partly compensated for by a different inferred prior or encoding function. Held-out negative log-likelihood (NLL), stability across folds, and scientific plausibility should be considered together.

> This workflow supports model exploration, not automatic scientific validation. A small NLL difference or a single fold is not decisive, and a final analysis should use enough trials and all folds.

## 1. Set up

Run the next cell. In Colab it clones the repository. Locally it finds `code/Demo`. Missing pinned dependencies are installed automatically.

**Optional Colab GPU:** before running the notebook, use Colab's runtime settings to select a GPU hardware accelerator, then reconnect and run from the top. The setup output should say `Compute device available: cuda`. If no GPU is available—or your Colab GPU quota is exhausted—the tutorial still runs on CPU, but fitting takes longer.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
from datetime import datetime
from zipfile import ZIP_DEFLATED, ZipFile
import csv
import math
import subprocess
import sys

IN_COLAB = find_spec("google.colab") is not None

if IN_COLAB:
    REPO_DIR = Path("/content/bayesian-models-of-perception")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/m-hahn/bayesian-models-of-perception.git", str(REPO_DIR)],
            check=True,
        )
    DEMO_DIR = REPO_DIR / "code" / "Demo"
else:
    candidates = [Path.cwd(), Path.cwd() / "code" / "Demo", Path.cwd() / "Demo"]
    DEMO_DIR = next(
        (candidate.resolve() for candidate in candidates
         if (candidate / "run_behavioral_pipeline.py").exists()),
        None,
    )
    if DEMO_DIR is None:
        raise RuntimeError("Run this notebook from the repository or code/Demo directory.")

required = ["matplotlib", "numpy", "scipy", "torch"]
if any(find_spec(package) is None for package in required):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(DEMO_DIR.parent / "requirements.txt")],
        check=True,
    )

sys.path.insert(0, str(DEMO_DIR))
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import FileLink, Image, display
from run_behavioral_pipeline import (
    CONFIG, fitted_profile_roughness, plot_errors_by_condition,
    prepare_five_condition_example, prepare_included_example,
    read_fitted_profiles,
    run_pipeline, validate_csv,
)

print(f"Demo directory: {DEMO_DIR}")
print(f"Python: {sys.executable}")
print(f"Compute device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if IN_COLAB and not torch.cuda.is_available():
    print("Optional speedup: enable a GPU in Colab's runtime settings, reconnect, and run again from the top.")

## 2. Choose a dataset

For a first pass, leave `USE_INCLUDED_EXAMPLE = True`. The included N = 1000 circular dataset was simulated with a p = 2 decision loss, a uniform prior, and nonuniform encoding. It gives us known structure against which to compare the recovered fits.

For your own data, set it to `False` and choose the appropriate space. In Colab, the next cell opens an upload dialog. Locally, set `LOCAL_CSV_PATH`. A CSV needs `stimulus` and `response` columns; `condition` is optional. Circular values are in `[0, 360)` and interval values in `[0, 3]`. Conditions represent sensory-noise levels.

In [ ]:
USE_INCLUDED_EXAMPLE = True
SPACE = "circular"  # Change to "interval" for interval data.
LOCAL_CSV_PATH = None  # For local data, put the CSV path here.

In [ ]:
uploaded = None
if USE_INCLUDED_EXAMPLE:
    print("Using the included simulated dataset; no upload is needed.")
elif IN_COLAB:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
else:
    print("Using LOCAL_CSV_PATH.")

In [ ]:
UPLOAD_DIR = DEMO_DIR / "input" / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if USE_INCLUDED_EXAMPLE:
    if SPACE != "circular":
        raise ValueError("The included example is circular; set SPACE = 'circular'.")
    INPUT_CSV = prepare_included_example()
elif IN_COLAB:
    if not uploaded:
        raise RuntimeError("Upload a CSV in the previous cell first.")
    INPUT_CSV = Path(next(iter(uploaded))).resolve()
else:
    if LOCAL_CSV_PATH is None:
        raise RuntimeError("Set LOCAL_CSV_PATH before continuing.")
    INPUT_CSV = Path(LOCAL_CSV_PATH).expanduser().resolve()

summary = validate_csv(INPUT_CSV, SPACE, wrap_circular=(SPACE == "circular"))
print(f"Using: {INPUT_CSV}")
print(f"Validated {summary.row_count} rows across conditions {summary.conditions}")
print(f"Stimulus range: {summary.stimulus_range}")
print(f"Response range: {summary.response_range}")

## 3. Look at the observations before fitting

Run the next cell. Check that the axes and condition labels make sense. The right panel shows response error (`response - stimulus`), wrapped to `[-180, 180)` for circular data. Structure here is what the model will try to explain through its prior, encoding, sensory noise, motor noise, and assumed decision loss.

Do not interpret a prior for the whole stimulus space if the experiment sampled only a narrow part of that space.

In [ ]:
conditions, stimuli, responses = [], [], []
with INPUT_CSV.open(newline="") as in_file:
    reader = csv.DictReader(in_file)
    default_condition = CONFIG[SPACE]["default_condition"]
    for row in reader:
        raw_condition = row.get("condition", "")
        conditions.append(int(float(raw_condition)) if raw_condition else default_condition)
        stimuli.append(float(row["stimulus"]))
        responses.append(float(row["response"]))

conditions = np.asarray(conditions)
stimuli = np.asarray(stimuli)
responses = np.asarray(responses)
errors = responses - stimuli
if SPACE == "circular":
    errors = (errors + 180) % 360 - 180
    lower, upper = 0, 360
else:
    lower, upper = 0, 3

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for condition in sorted(set(conditions)):
    mask = conditions == condition
    axes[0].scatter(stimuli[mask], responses[mask], s=12, alpha=0.55, label=str(condition))
    axes[1].scatter(stimuli[mask], errors[mask], s=12, alpha=0.35)
axes[0].plot([lower, upper], [lower, upper], color="black", linewidth=1)
axes[0].set(xlabel="stimulus", ylabel="response", title="Responses")
axes[0].legend(title="condition", frameon=False)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="stimulus", ylabel="signed response error", title="Empirical error pattern")
fig.tight_layout()
plt.show()

## 4. Configure the experiment sequence

Run the next cell as written for a tutorial-speed analysis. The notebook uses one regularization weight until the later regularization section. `QUICK_MODE` uses a coarser grid and three folds. These results are provisional. Set it to `False` for the default grid and ten folds; that can take substantially longer, especially on CPU. Leave `USE_GPU_IF_AVAILABLE = True` to use a Colab or local CUDA GPU automatically, or set it to `False` to force CPU execution.

The candidate `p` values describe the participant's assumed decision loss, not the fitting objective:

- `p = 0`: MAP-like decision rule.
- `p = 1`: absolute-error-like decision rule.
- `p = 2`: squared-error/cosine-loss analogue.
- larger even `p`: increasingly strong penalties for large decision errors.

Every candidate is fitted by likelihood and compared by held-out NLL. The regularization weight penalizes adjacent changes in the prior and encoding logits: too little can fit noise; too much can erase genuine structure.

In [ ]:
QUICK_MODE = True
USE_GPU_IF_AVAILABLE = True
BASELINE_P = 2
REG_WEIGHT = 10.0

if QUICK_MODE:
    GRID = 60 if SPACE == "circular" else 80
    EVALUATION_FOLDS = [0, 1, 2]
else:
    GRID = None  # 180 circular; 400 interval
    EVALUATION_FOLDS = list(range(10))

DEVICE = "cuda" if USE_GPU_IF_AVAILABLE and torch.cuda.is_available() else "cpu"
PLOT_EVERY = 1000
RUN_NAME = f"tutorial-{INPUT_CSV.stem}-{datetime.now():%Y%m%d-%H%M%S}"
print(f"Run name: {RUN_NAME}")
print(f"Device: {DEVICE}; grid: {GRID or 'model default'}")
print(f"Regularization weight: {REG_WEIGHT}")
print(f"Evaluation folds: {EVALUATION_FOLDS}")

### Helper functions

Run this cell once. Fitting uses `run_pipeline` directly in the sections below. These small helpers only keep track of completed runs and display their results.

In [ ]:
PIPELINE_RESULTS = []

def plot_fitted_profiles(fits, labels, title, space=SPACE):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
    for fit, label in zip(fits, labels):
        profiles = read_fitted_profiles(fit, space)
        axes[0].plot(profiles["grid"], profiles["prior"], label=label)
        axes[1].plot(profiles["grid"], profiles["encoding"], label=label)
        edges = np.linspace(0, profiles["upper"], len(profiles["encoding"]) + 1)
        cumulative = np.r_[0, np.cumsum(profiles["encoding"] / profiles["encoding"].sum())]
        axes[2].plot(edges, cumulative, label=label)
    axes[0].set(title="Inferred prior", xlabel="stimulus", ylabel="relative density (mean 1)", ylim=(0, None))
    axes[1].set(title="Encoding allocation", xlabel="stimulus", ylabel="relative resources (mean 1)", ylim=(0, None))
    axes[2].set(title="Cumulative encoding map", xlabel="stimulus", ylabel="normalized sensory coordinate")
    for axis in axes:
        axis.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

def show_artifacts(fit):
    print(f"NLL: {fit.cross_validation_loss:.6g}")
    print(f"Loss file: {fit.loss_path}")
    print(f"Parameter log: {fit.parameter_log_path}")
    if not IN_COLAB:
        display(FileLink(str(fit.loss_path)))
        display(FileLink(str(fit.parameter_log_path)))
    if fit.figure_path is not None and fit.figure_path.exists():
        print(f"Diagnostic PDF: {fit.figure_path}")
        if not IN_COLAB:
            display(FileLink(str(fit.figure_path)))
        if fit.figure_preview_path.exists():
            display(Image(filename=str(fit.figure_preview_path)))
        else:
            print("PNG preview unavailable; rerun this fit to create it.")

print("Helpers ready.")

## 5. Establish a baseline

Start with `p = 2`, regularization weight 10, and fold 0. This verifies that the complete fitting and output path works before launching a sweep. Run both cells below.

In [ ]:
baseline_result = run_pipeline(
    INPUT_CSV,
    SPACE,
    p=[BASELINE_P],
    fold=0,
    reg_weight=REG_WEIGHT,
    grid=GRID,
    dataset_name=RUN_NAME,
    wrap_circular=(SPACE == "circular"),
    device=DEVICE,
    plot_every=PLOT_EVERY,
    quiet=True,
    overwrite=True,
)
PIPELINE_RESULTS.append(baseline_result)
baseline_fit = baseline_result.fits[0]

In [ ]:
show_artifacts(baseline_fit)
plot_fitted_profiles([baseline_fit], ["p=2, lambda=10"], "Baseline inferred structure")

Before continuing, open the diagnostic PDF. For circular fits it includes prior, encoding resources, predicted attraction/repulsion and bias, the empirical bias, and variability. The custom plots above show the saved prior and encoding on a common scale. Values above 1 indicate more prior density or encoding resources than the uniform allocation.

## 6. Compare decision-loss functions

First compare `p = 2` and `p = 8`, using exactly the same held-out fold, regularization, and grid. The included example was generated with `p = 2`, so `p = 2` is the known truth there. In one quick-mode CPU reference run, fold-0 NLL was about `44.02` at `p = 2` and `48.32` at `p = 8`; small numerical differences are expected. After inspecting the result, edit `P_CANDIDATES` in the following cell to choose any additional values you want to try.

In [ ]:
coarse_result = run_pipeline(
    INPUT_CSV,
    SPACE,
    p=[8],
    fold=0,
    reg_weight=REG_WEIGHT,
    grid=GRID,
    dataset_name=RUN_NAME,
    wrap_circular=(SPACE == "circular"),
    device=DEVICE,
    plot_every=PLOT_EVERY,
    quiet=True,
    overwrite=True,
)
PIPELINE_RESULTS.append(coarse_result)
loss_fits = {BASELINE_P: baseline_fit, 8: coarse_result.fits[0]}
coarse_ranking = sorted(
    [(p, loss_fits[p].cross_validation_loss) for p in [2, 8]],
    key=lambda item: item[1],
)
print("\nCoarse p=2 versus p=8 screen:")
for p, nll in coarse_ranking:
    print(f"p={p}: NLL={nll:.6g}, delta={nll - coarse_ranking[0][1]:.6g}")

In [ ]:
P_CANDIDATES = [0, 1, 4]  # Edit this list to try other loss exponents.
candidate_result = run_pipeline(
    INPUT_CSV,
    SPACE,
    p=P_CANDIDATES,
    fold=0,
    reg_weight=REG_WEIGHT,
    grid=GRID,
    dataset_name=RUN_NAME,
    wrap_circular=(SPACE == "circular"),
    device=DEVICE,
    plot_every=PLOT_EVERY,
    quiet=True,
    overwrite=True,
)
PIPELINE_RESULTS.append(candidate_result)
for fit in candidate_result.fits:
    loss_fits[fit.p] = fit

loss_ranking = sorted(
    [(p, fit.cross_validation_loss) for p, fit in loss_fits.items()],
    key=lambda item: item[1],
)
BEST_P = loss_ranking[0][0]
print("Adaptive fold-0 loss screen (lower NLL is better):")
for rank, (p, nll) in enumerate(loss_ranking, start=1):
    print(f"{rank:>2}. p={p:<2} NLL={nll:.6g}  delta={nll - loss_ranking[0][1]:.6g}")

fig, axis = plt.subplots(figsize=(6, 3.5))
axis.bar([str(p) for p, _ in loss_ranking], [nll for _, nll in loss_ranking])
axis.set(xlabel="decision-loss exponent p", ylabel="held-out NLL", title="Loss-function screen: fold 0")
axis.spines[["top", "right"]].set_visible(False)
plt.show()
print(f"Provisional best loss: p={BEST_P}")

## 7. See what the wrong loss does to interpretation

For the included example, the next cell contrasts the known generating loss (`p = 2`) with a deliberately worse candidate. For your own dataset, where the truth is unknown, it contrasts the best and worst candidates from the provisional first-fold screen. Look for movement of peaks, changes in smoothness, or a tradeoff between prior and encoding. Those changes show how a misspecified decision rule can be absorbed into supposedly perceptual components.

In [ ]:
REFERENCE_P = BEST_P
if USE_INCLUDED_EXAMPLE:
    REFERENCE_P = 2
WORST_P = loss_ranking[-1][0]
if WORST_P == REFERENCE_P:
    WORST_P = loss_ranking[-2][0]
reference_loss_fit = loss_fits[REFERENCE_P]
wrong_loss_fit = loss_fits[WORST_P]
plot_fitted_profiles(
    [reference_loss_fit, wrong_loss_fit],
    [f"reference: p={REFERENCE_P}", f"mismatched: p={WORST_P}"],
    "Consequences of the assumed loss function",
)
reference_profiles = read_fitted_profiles(reference_loss_fit, SPACE)
wrong_profiles = read_fitted_profiles(wrong_loss_fit, SPACE)
prior_rms = np.sqrt(np.mean((reference_profiles["prior"] - wrong_profiles["prior"]) ** 2))
encoding_rms = np.sqrt(np.mean((reference_profiles["encoding"] - wrong_profiles["encoding"]) ** 2))
print(f"Held-out NLL difference (p={WORST_P} minus p={REFERENCE_P}): {wrong_loss_fit.cross_validation_loss - reference_loss_fit.cross_validation_loss:.6g}")
print(f"RMS change in relative prior: {prior_rms:.4g}")
print(f"RMS change in relative encoding allocation: {encoding_rms:.4g}")
print("\nReference fit artifacts:")
show_artifacts(reference_loss_fit)
print("\nMismatched-loss fit artifacts:")
show_artifacts(wrong_loss_fit)

Pause and answer these questions before moving on:

1. Does the worse-loss fit have a clearly larger held-out NLL, or is the difference small?
2. Which prior features move or disappear?
3. Which encoding regions gain or lose resources?
4. If NLLs are similar but the inferred components differ, is the dataset informative enough to separate loss, prior, and encoding?

Do not choose a loss because its inferred curves look nicer. Use held-out fit first, then ask whether the result is stable and scientifically credible.

## 8. Repeat the anchor comparison across five noise levels

A single noise level can make the loss, prior, encoding, and sensory noise hard to distinguish. The repository also includes a larger synthetic dataset with five conditions, generated with `p = 8`. In this exercise, the prior and encoding are shared across conditions while the model estimates a separate sensory-noise parameter for each condition. That added variation can help expose a misspecified loss.

Run the next cell to prepare and inspect this second dataset. This is a controlled teaching example, not a replacement for the analysis of your own data above.

In [ ]:
MULTI_INPUT_CSV = prepare_five_condition_example()
multi_summary = validate_csv(MULTI_INPUT_CSV, "circular", wrap_circular=True)
multi_condition_ids = list(multi_summary.conditions)
figure, axes = plot_errors_by_condition(MULTI_INPUT_CSV, "circular")
plt.show()
print(f"Validated {multi_summary.row_count} trials across conditions {multi_summary.conditions}.")

Now fit only the two coarse anchors, `p = 2` and `p = 8`. Quick mode deliberately uses a coarse grid to keep the exercise manageable. Compare NLLs only within this dataset and held-out fold: its absolute NLL cannot be compared with the earlier dataset because the number of observations differs.

In [ ]:
MULTI_GRID = 40 if QUICK_MODE else 180
MULTI_RUN_NAME = f"{RUN_NAME}-five-noise-p8-truth"
multi_result = run_pipeline(
    MULTI_INPUT_CSV,
    "circular",
    p=[2, 8],
    fold=0,
    reg_weight=REG_WEIGHT,
    grid=MULTI_GRID,
    dataset_name=MULTI_RUN_NAME,
    wrap_circular=True,
    device=DEVICE,
    plot_every=PLOT_EVERY,
    quiet=True,
    overwrite=True,
)
PIPELINE_RESULTS.append(multi_result)
MULTI_FITS = {fit.p: fit for fit in multi_result.fits}

multi_best_p = min(MULTI_FITS, key=lambda p: MULTI_FITS[p].cross_validation_loss)
for p in [2, 8]:
    print(f"p={p}: held-out NLL={MULTI_FITS[p].cross_validation_loss:.6g}")
print(f"NLL difference (p=2 minus p=8): {MULTI_FITS[2].cross_validation_loss - MULTI_FITS[8].cross_validation_loss:.6g}")
print(f"Preferred anchor on this fold: p={multi_best_p}; known generating value: p=8.")
if multi_best_p != 8:
    print("The quick fit did not recover p=8. Re-run in full mode and across folds before interpreting this as evidence against the generating loss.")

In [ ]:
plot_fitted_profiles(
    [MULTI_FITS[2], MULTI_FITS[8]],
    ["mismatched p=2", "generating p=8"],
    "Five noise levels: consequences of the assumed loss",
    space="circular",
)

fig, axis = plt.subplots(figsize=(6.5, 3.8))
for p, marker in [(2, "o"), (8, "s")]:
    profiles = read_fitted_profiles(MULTI_FITS[p], "circular")
    sigma_squared = 4 / (1 + np.exp(-profiles["raw"]["sigma_logit"]))
    axis.plot(profiles["condition_ids"], sigma_squared, marker=marker, label=f"p={p}")
axis.set(xticks=multi_condition_ids, xlabel="condition", ylabel=r"fitted sensory-noise parameter $\sigma^2$", title="Condition-specific sensory noise")
axis.spines[["top", "right"]].set_visible(False)
axis.legend(frameon=False)
fig.tight_layout()
plt.show()

for p in [2, 8]:
    print(f"\nFive-noise-level p={p} artifacts:")
    show_artifacts(MULTI_FITS[p])

In a quick-mode CPU reference run, fold-0 NLL was about `474.14` at the mismatched `p = 2` and `456.03` at the generating `p = 8`; small numerical differences are expected. The comparison illustrates how a misspecified `p` can be compensated for through changes in the inferred prior.

## 9. Tune regularization for the provisional loss

This section returns to the primary dataset selected in Section 2; it does not use the separate five-condition example. Set `P_FOR_REGULARIZATION` explicitly, then vary only the regularization weight. The included primary example was generated with `p = 2`; for another dataset, use the preferred value from the loss comparison above. The validation NLL does **not** include the smoothness penalty, so it measures held-out predictive fit. The roughness measure plotted below describes the under-smoothed versus over-smoothed tradeoff.

In [ ]:
P_FOR_REGULARIZATION = 2  # Change this after selecting p for another dataset.
REG_CANDIDATES = [0.3, 3.0, 10.0, 30.0]
reg_fits = {}
for reg_weight in REG_CANDIDATES:
    reg_result = run_pipeline(
        INPUT_CSV,
        SPACE,
        p=[P_FOR_REGULARIZATION],
        fold=0,
        reg_weight=reg_weight,
        grid=GRID,
        dataset_name=RUN_NAME,
        wrap_circular=(SPACE == "circular"),
        device=DEVICE,
        plot_every=PLOT_EVERY,
        quiet=True,
        overwrite=True,
    )
    PIPELINE_RESULTS.append(reg_result)
    reg_fits[reg_weight] = reg_result.fits[0]

reg_ranking = sorted(
    [(reg, fit.cross_validation_loss) for reg, fit in reg_fits.items()],
    key=lambda item: item[1],
)
BEST_REG_WEIGHT = reg_ranking[0][0]
print("Fold-0 regularization screen:")
for rank, (reg, nll) in enumerate(reg_ranking, start=1):
    prior_roughness, encoding_roughness = fitted_profile_roughness(reg_fits[reg], SPACE)
    print(
        f"{rank:>2}. lambda={reg:<6g} NLL={nll:.6g}  "
        f"prior roughness={prior_roughness:.3g}  encoding roughness={encoding_roughness:.3g}"
    )
print(f"Provisional best regularization: {BEST_REG_WEIGHT:g}")

In [ ]:
ordered_regs = sorted(reg_fits)
nlls = [reg_fits[reg].cross_validation_loss for reg in ordered_regs]
roughness_values = [fitted_profile_roughness(reg_fits[reg], SPACE) for reg in ordered_regs]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ordered_regs, nlls, marker="o")
axes[0].set_xscale("log")
axes[0].set(xlabel="regularization weight lambda", ylabel="held-out NLL", title="Predictive fit")
axes[1].plot(ordered_regs, [value[0] for value in roughness_values], marker="o", label="prior")
axes[1].plot(ordered_regs, [value[1] for value in roughness_values], marker="o", label="encoding")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set(xlabel="regularization weight lambda", ylabel="profile roughness", title="Smoothness tradeoff")
axes[1].legend(frameon=False)
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

plot_fitted_profiles(
    [reg_fits[reg] for reg in ordered_regs],
    [f"lambda={reg:g}" for reg in ordered_regs],
    f"Regularization sweep at p={P_FOR_REGULARIZATION}",
)

A useful regularization choice predicts held-out observations well without producing fold-specific oscillations. Very smooth curves are not automatically correct, and very detailed curves are not automatically informative. Regularization weights are tied to the chosen grid and parameterization, so do not copy a winning number blindly to a different analysis.

## 10. Confirm a shortlist across folds

The fold-0 screens were deliberately cheap. The next cell constructs a shortlist containing the provisional joint winner, the runner-up regularization at that loss, and the runner-up loss at the baseline regularization. Inspect `SHORTLIST`; you may edit it before running the following cell.

Quick mode evaluates three folds. A serious report should set `QUICK_MODE = False` above and evaluate all ten. Screening then confirming a shortlist is pragmatic, but it is not a substitute for nested cross-validation when unbiased hyperparameter-selection performance is required.

In [ ]:
SHORTLIST = [
    (P_FOR_REGULARIZATION, BEST_REG_WEIGHT),
    (P_FOR_REGULARIZATION, reg_ranking[1][0]),
    (loss_ranking[1][0], REG_WEIGHT),
]
print("Configurations to confirm (p, lambda):")
for configuration in SHORTLIST:
    print(" ", configuration)

In [ ]:
confirmed_fits = {}
for p, reg_weight in SHORTLIST:
    fits = []
    for fold in EVALUATION_FOLDS:
        fold_result = run_pipeline(
            INPUT_CSV,
            SPACE,
            p=[p],
            fold=fold,
            reg_weight=reg_weight,
            grid=GRID,
            dataset_name=RUN_NAME,
            wrap_circular=(SPACE == "circular"),
            device=DEVICE,
            plot_every=PLOT_EVERY,
            quiet=True,
            overwrite=True,
        )
        PIPELINE_RESULTS.append(fold_result)
        fits.append(fold_result.fits[0])
    confirmed_fits[(p, reg_weight)] = fits

In [ ]:
confirmation_rows = []
for (p, reg_weight), fits in confirmed_fits.items():
    values = np.asarray([fit.cross_validation_loss for fit in fits])
    confirmation_rows.append((p, reg_weight, values.mean(), values))
confirmation_rows.sort(key=lambda row: row[2])
WINNING_P, WINNING_REG_WEIGHT = confirmation_rows[0][:2]
reference_values = confirmation_rows[0][3]

print("Multi-fold ranking (lower mean NLL is better):")
paired_differences = []
paired_errors = []
for rank, (p, reg_weight, mean_nll, values) in enumerate(confirmation_rows, start=1):
    differences = values - reference_values
    difference_se = differences.std(ddof=1) / math.sqrt(len(differences)) if len(differences) > 1 else float("nan")
    paired_differences.append(differences.mean())
    paired_errors.append(difference_se)
    print(
        f"{rank:>2}. p={p:<2} lambda={reg_weight:<6g} mean NLL={mean_nll:.6g}  "
        f"paired delta={differences.mean():.4g} (SE {difference_se:.3g})  folds={values.tolist()}"
    )

labels = [f"p={p}, lambda={reg:g}" for p, reg, *_ in confirmation_rows]
fig, axis = plt.subplots(figsize=(max(7, 2.2 * len(labels)), 4))
axis.errorbar(range(len(labels)), paired_differences, yerr=paired_errors, fmt="o", capsize=4)
axis.axhline(0, color="black", linewidth=1)
axis.set_xticks(range(len(labels)), labels, rotation=20, ha="right")
axis.set(ylabel="paired NLL difference from winner", title="Shortlist confirmation across folds")
axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()
print(f"Current winner: p={WINNING_P}, lambda={WINNING_REG_WEIGHT:g}")
if USE_INCLUDED_EXAMPLE:
    print("Known generating loss for this example: p=2.")
    if WINNING_P != 2:
        print("Quick mode did not recover it; treat this as a warning to use the full grid, all folds, and adequate data before drawing conclusions.")

The plot uses paired fold-by-fold NLL differences, which removes variation shared by all configurations on an unusually easy or difficult fold. Do not over-read the rank order. If uncertainty bars overlap zero or the NLL advantage is tiny, report that the data do not distinguish those configurations clearly. Prefer a conclusion that survives reasonable grids, regularization ranges, and folds.

## 11. Interpret the winning prior and encoding—and check stability

The thin lines below are separate folds; the thick line is their mean. Stable peaks are more credible than features appearing in only one fold. The prior describes relative probability before the current sensory observation. The encoding allocation describes where representational resources are concentrated; its cumulative sum is the nonlinear map into normalized sensory coordinates. Neither should be interpreted independently of the selected loss.

In [ ]:
winner_fits = confirmed_fits[(WINNING_P, WINNING_REG_WEIGHT)]
winner_profiles = [read_fitted_profiles(fit, SPACE) for fit in winner_fits]
grid = winner_profiles[0]["grid"]
prior_stack = np.stack([profile["prior"] for profile in winner_profiles])
encoding_stack = np.stack([profile["encoding"] for profile in winner_profiles])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for fold, prior, encoding in zip(EVALUATION_FOLDS, prior_stack, encoding_stack):
    axes[0].plot(grid, prior, color="C0", alpha=0.25, linewidth=1, label=f"fold {fold}")
    axes[1].plot(grid, encoding, color="C1", alpha=0.25, linewidth=1, label=f"fold {fold}")
axes[0].plot(grid, prior_stack.mean(axis=0), color="C0", linewidth=3, label="fold mean")
axes[1].plot(grid, encoding_stack.mean(axis=0), color="C1", linewidth=3, label="fold mean")
axes[0].set(title="Winning prior across folds", xlabel="stimulus", ylabel="relative density", ylim=(0, None))
axes[1].set(title="Winning encoding across folds", xlabel="stimulus", ylabel="relative resources", ylim=(0, None))
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

print(f"Condition order in the parameter logs: {winner_profiles[0]['condition_ids']}")
print("The sigma_logit entries use this same condition order.")
for fold, fit in zip(EVALUATION_FOLDS, winner_fits):
    print(f"\nWinning fit, fold {fold}:")
    show_artifacts(fit)

### Interpretation checklist

Before making a substantive claim, write down answers to these questions:

1. Which `(p, regularization)` configurations are genuinely separated by held-out NLL?
2. Does the preferred `p` remain preferred across folds and nearby regularization values?
3. Which prior and encoding features recur across folds?
4. Did a worse loss produce a different but plausible-looking prior or encoding? If so, explicitly report that sensitivity.
5. Are inferred features supported where stimuli were actually sampled?
6. Would the conclusion survive the full grid, all ten folds, and more than one optimization start?

For the included synthetic example, compare your result with its known p = 2 loss, uniform prior, and nonuniform encoding. Recovery need not be exact in quick mode. For a real dataset there is no known answer, so predictive evidence and stability matter more.

## 12. Save the experiment

Run the final cell to write a CSV containing every completed fit—including the five-noise-level exercise—and bundle it with both inputs, converted model data, NLL files, parameter logs, and figures.

In [ ]:
completed = {}
for result in PIPELINE_RESULTS:
    for fit in result.fits:
        completed[fit.parameter_log_path] = (result, fit)

SUMMARY_CSV = UPLOAD_DIR / f"{RUN_NAME}-model-selection.csv"
with SUMMARY_CSV.open("w", newline="") as out_file:
    writer = csv.writer(out_file)
    writer.writerow(["dataset", "p", "regularization", "fold", "grid", "cross_validation_nll", "loss_file", "parameter_log", "figure"])
    for result, fit in completed.values():
        writer.writerow([
            result.dataset.input_csv.stem, fit.p, fit.reg_weight, fit.fold, fit.grid,
            fit.cross_validation_loss,
            fit.loss_path, fit.parameter_log_path, fit.figure_path or "",
        ])

RESULTS_ZIP = DEMO_DIR / f"{RUN_NAME}-fit-results.zip"
with ZipFile(RESULTS_ZIP, "w", ZIP_DEFLATED) as archive:
    archive.write(INPUT_CSV, f"input/{INPUT_CSV.name}")
    archive.write(MULTI_INPUT_CSV, f"input/{MULTI_INPUT_CSV.name}")
    archive.write(SUMMARY_CSV, f"summary/{SUMMARY_CSV.name}")
    for converted_path in {result.legacy_dataset_path for result in PIPELINE_RESULTS}:
        archive.write(converted_path, f"converted/{converted_path.name}")
    added = set()
    for result, fit in completed.values():
        for path in (fit.loss_path, fit.parameter_log_path, fit.figure_path, fit.figure_preview_path):
            if path is not None and path.exists() and path not in added:
                archive.write(path, f"results/{path.name}")
                added.add(path)

print(f"Summary table: {SUMMARY_CSV}")
print(f"Result bundle: {RESULTS_ZIP}")
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(RESULTS_ZIP))
else:
    display(FileLink(str(RESULTS_ZIP)))

## Where to go next

Repeat the analysis with `QUICK_MODE = False`, consider additional optimization starts, and pre-specify the final comparison before reporting results. Keep the NLL files, parameter logs, and PDFs together—the apparent prior or encoding is meaningful only in the context of its loss, regularization, grid, fold, and data coverage.

Please do not hesitate at all to contact Michael Hahn at [mhahn@lst.uni-saarland.de](mailto:mhahn@lst.uni-saarland.de) with any questions. He is very happy to provide advice or help troubleshoot. You can also open a repository issue.